# 09 — Runtime com dataclasses

**Trilha:** [07 contrato](07_contrato_e_contexto.ipynb) → [08 políticas](08_capacidades_e_politicas.ipynb) → **09 runtime com dataclasses** → [10 pipeline](10_pipeline_completo.ipynb)

**De onde viemos:** o [08](08_capacidades_e_politicas.ipynb) terminou com um fluxo seguro, mas opaco: não sabemos quantas
chamadas fez, quantos tokens gastou, quanto custou nem quanto demorou. E o [05](../02_intermediario/05_regras_e_retry.ipynb) já
mostrou a conversa crescendo a cada rodada.

Aqui entra o **estado de execução**: dados internos do programa, que não vêm de fora e não precisam de validação a cada
mudança. Esse é o terreno da `dataclass` — Pydantic fica na fronteira (o que entra e sai do modelo).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from prettifier import show, show_table

In [2]:
import json
from contextlib import contextmanager
from dataclasses import dataclass, field
from decimal import Decimal
from time import perf_counter

from llm_simulado import ClienteSimulado, estimar_tokens, pedido_de_ferramenta

## 1. Configuração, uso e custo

In [3]:
@dataclass(frozen=True)
class ConfigRuntime:
    modelo: str = "gpt-4o-mini"
    max_rodadas: int = 5
    orcamento_tokens_envio: int = 70
    # US$ por milhão de tokens — valores ILUSTRATIVOS; use a tabela do seu provedor
    preco_entrada_por_milhao: Decimal = Decimal("0.15")
    preco_saida_por_milhao: Decimal = Decimal("0.60")


@dataclass
class Uso:
    chamadas: int = 0
    tokens_entrada: int = 0
    tokens_saida: int = 0

    def registrar(self, usage) -> None:
        self.chamadas += 1
        self.tokens_entrada += usage.prompt_tokens
        self.tokens_saida += usage.completion_tokens

    def custo(self, config: ConfigRuntime) -> Decimal:
        milhao = Decimal(1_000_000)
        return (self.tokens_entrada * config.preco_entrada_por_milhao
                + self.tokens_saida * config.preco_saida_por_milhao) / milhao

## 2. Medindo tempo

Um `Span` por etapa, aberto e fechado por um *context manager*. `perf_counter` é o relógio certo para medir duração (é
monotônico; `time.time()` pode pular quando o relógio do sistema é ajustado).

In [4]:
@dataclass
class Span:
    nome: str
    inicio: float
    fim: float | None = None

    @property
    def duracao_ms(self) -> float:
        return ((self.fim or perf_counter()) - self.inicio) * 1000


@dataclass
class Rastreio:
    spans: list[Span] = field(default_factory=list)

    @contextmanager
    def medir(self, nome: str):
        span = Span(nome, perf_counter())
        self.spans.append(span)
        try:
            yield span
        finally:
            span.fim = perf_counter()

## 3. Histórico com orçamento de tokens

Enviar a conversa inteira toda vez fica caro e, um dia, passa do limite do modelo. O histórico guarda tudo, mas **envia** só o
que cabe no orçamento: o prompt de sistema e os **turnos** mais recentes.

Um turno começa numa mensagem do usuário e inclui tudo até a próxima: as chamadas de ferramenta do assistente **e** os
resultados. Cortar no meio de um turno separaria um resultado da chamada que o pediu (pegadinha 1).

In [5]:
def tokens_da_mensagem(mensagem: dict) -> int:
    return estimar_tokens(str(mensagem.get("content") or "") + json.dumps(mensagem.get("tool_calls") or ""))


@dataclass
class Historico:
    sistema: dict
    mensagens: list[dict] = field(default_factory=list)

    def turnos(self) -> list[list[dict]]:
        turnos: list[list[dict]] = []
        for mensagem in self.mensagens:
            if mensagem["role"] == "user" or not turnos:
                turnos.append([])
            turnos[-1].append(mensagem)
        return turnos

    def para_envio(self, orcamento_tokens: int) -> list[dict]:
        restante = orcamento_tokens - tokens_da_mensagem(self.sistema)
        escolhidos: list[list[dict]] = []
        for turno in reversed(self.turnos()):
            custo = sum(tokens_da_mensagem(m) for m in turno)
            if custo > restante and escolhidos:
                break
            escolhidos.insert(0, turno)
            restante -= custo
        return [self.sistema] + [m for turno in escolhidos for m in turno]

## 4. O agente: tudo junto

In [6]:
PRECOS = {"azeite": "45.00", "tomate italiano": "1.20", "muçarela de búfala": "89.90"}


def executar_ferramenta(nome: str, argumentos: str) -> str:
    if nome != "consultar_preco":
        return json.dumps({"erro": f"ferramenta desconhecida: {nome}"})
    item = json.loads(argumentos).get("nome", "")
    return json.dumps({"nome": item, "preco": PRECOS.get(item)}, ensure_ascii=False)


@dataclass
class Agente:
    cliente: object
    config: ConfigRuntime = field(default_factory=ConfigRuntime)
    historico: Historico = field(default_factory=lambda: Historico({"role": "system", "content": "Assistente de compras."}))
    uso: Uso = field(default_factory=Uso)
    rastreio: Rastreio = field(default_factory=Rastreio)

    def perguntar(self, texto: str) -> str:
        self.historico.mensagens.append({"role": "user", "content": texto})
        with self.rastreio.medir(f"pergunta: {texto[:30]}"):
            for rodada in range(1, self.config.max_rodadas + 1):
                enviadas = self.historico.para_envio(self.config.orcamento_tokens_envio)
                with self.rastreio.medir(f"  modelo (rodada {rodada}, {len(enviadas)} msgs)"):
                    resposta = self.cliente.chat.completions.create(model=self.config.modelo, messages=enviadas)
                self.uso.registrar(resposta.usage)
                mensagem = resposta.choices[0].message
                self.historico.mensagens.append(mensagem.model_dump(exclude_none=True))
                if not mensagem.tool_calls:
                    return mensagem.content
                for chamada in mensagem.tool_calls:
                    with self.rastreio.medir(f"  ferramenta {chamada.function.name}"):
                        resultado = executar_ferramenta(chamada.function.name, chamada.function.arguments)
                    self.historico.mensagens.append({"role": "tool", "tool_call_id": chamada.id, "content": resultado})
        raise RuntimeError(f"sem resposta em {self.config.max_rodadas} rodadas")

Três perguntas seguidas (*roteiro simulado*), com um orçamento de envio pequeno de propósito para o corte aparecer:

In [7]:
roteiro = [
    pedido_de_ferramenta(("consultar_preco", {"nome": "azeite"})),
    "O azeite está R$ 45,00 o litro.",
    pedido_de_ferramenta(("consultar_preco", {"nome": "tomate italiano"})),
    "O tomate italiano está R$ 1,20 a unidade.",
    pedido_de_ferramenta(("consultar_preco", {"nome": "muçarela de búfala"})),
    "A muçarela de búfala está R$ 89,90 o quilo.",
]
cliente = ClienteSimulado(roteiro)
agente = Agente(cliente)
for pergunta in ["Quanto está o azeite?", "E o tomate italiano?", "E a muçarela de búfala?"]:
    show(f"{pergunta} -> {agente.perguntar(pergunta)}")

Quanto está o azeite? -> O azeite está R$ 45,00 o litro.
E o tomate italiano? -> O tomate italiano está R$ 1,20 a unidade.
E a muçarela de búfala? -> A muçarela de búfala está R$ 89,90 o quilo.


In [8]:
show(f"mensagens no histórico: {len(agente.historico.mensagens)}")
show(f"mensagens enviadas por chamada: {[len(c['messages']) for c in cliente.chamadas]}")
show(f"agente.uso: {agente.uso}")
show(f"custo: US$ {agente.uso.custo(agente.config):.8f}")
show_table(["etapa", "duração (ms)"], [(span.nome, f"{span.duracao_ms:8.3f}") for span in agente.rastreio.spans],
       title="rastreio")

mensagens no histórico: 12
mensagens enviadas por chamada: [2, 4, 6, 4, 6, 4]
agente.uso: Uso(chamadas=6, tokens_entrada=309, tokens_saida=124)
custo: US$ 0.00012075
rastreio                                            
                                                    
  etapa                               duração (ms)  
 ────────────────────────────────────────────────── 
  pergunta: Quanto está o azeite?        3.501      
    modelo (rodada 1, 2 msgs)            3.347      
    ferramenta consultar_preco           0.024      
    modelo (rodada 2, 4 msgs)            0.038      
  pergunta: E o tomate italiano?         0.299      
    modelo (rodada 1, 6 msgs)            0.121      
    ferramenta consultar_preco           0.023      
    modelo (rodada 2, 4 msgs)            0.046      
  pergunta: E a muçarela de búfala?      0.229      
    modelo (rodada 1, 6 msgs)            0.073      
    ferramenta consultar_preco           0.021      
    modelo (rodada 2, 4 msgs)          

O histórico guarda todas as mensagens; cada chamada enviou só os turnos que cabiam no orçamento — o turno atual sempre vai. As durações são de um modelo simulado (quase
zero); com a API real, o tempo do modelo domina tudo.

## ⚠️ O que ainda quebra

**1. Cortar por número de mensagens quebra os pares de ferramenta.** Um corte ingênuo ("as últimas N mensagens") pode começar
num resultado de ferramenta cuja chamada ficou para trás. A API da OpenAI recusa conversas assim (erro 400). Conferindo:

In [9]:
def pares_consistentes(mensagens: list[dict]) -> bool:
    pedidos = set()
    for m in mensagens:
        for chamada in m.get("tool_calls") or []:
            pedidos.add(chamada["id"])
        if m["role"] == "tool" and m["tool_call_id"] not in pedidos:
            return False
    return True


ingenuo = [agente.historico.sistema] + agente.historico.mensagens[-2:]
show(f"corte ingênuo começa com: {[m['role'] for m in ingenuo]}")
show(f"pares_consistentes(corte ingênuo): {pares_consistentes(ingenuo)}")
show(f"pares_consistentes(todas as chamadas do agente): {all(pares_consistentes(c['messages']) for c in cliente.chamadas)}")

corte ingênuo começa com: ['system', 'tool', 'assistant']
pares_consistentes(corte ingênuo): False
pares_consistentes(todas as chamadas do agente): True


**2. Lista como default de dataclass.** O erro clássico é bloqueado na criação da classe — o remédio é `field(default_factory=list)`,
usado em `Historico`, `Rastreio` e `Agente`:

In [10]:
try:
    @dataclass
    class HistoricoErrado:
        mensagens: list = []
except ValueError as ex:
    show(f"ValueError: {ex}")

ValueError: mutable default <class 'list'> for field mensagens is not allowed: use default_factory


**3. `Decimal` não se mistura com `float`.** Somar ou multiplicar os dois levanta erro — o que é bom, porque obriga a decidir.
O que chega como `float` (de JSON, por exemplo) entra via `Decimal(str(x))`:

In [11]:
try:
    Decimal("0.15") * 0.5
except TypeError as ex:
    show(f"TypeError: {ex}")
show(f"Decimal('0.15') * Decimal(str(0.5)): {Decimal('0.15') * Decimal(str(0.5))}")

TypeError: unsupported operand type(s) for *: 'decimal.Decimal' and 'float'
Decimal('0.15') * Decimal(str(0.5)): 0.075


**4. A contagem de tokens aqui é estimativa.** O corte do histórico precisa decidir **antes** de enviar, então usa a estimativa
(1 token ≈ 4 caracteres). O número real vem depois, em `resposta.usage` — e é ele que deve ir para o custo. Para cortar com
precisão, use o tokenizer do modelo (por exemplo, a biblioteca `tiktoken` para modelos da OpenAI).

**5. Dataclass não valida.** `ConfigRuntime(max_rodadas="cinco")` é aceito sem reclamar — e quebra só no `range`, longe da
causa:

In [12]:
config_ruim = ConfigRuntime(max_rodadas="cinco")
show(f"config_ruim.max_rodadas: {config_ruim.max_rodadas!r}")
try:
    range(1, config_ruim.max_rodadas + 1)
except TypeError as ex:
    show(f"TypeError (longe da causa): {ex}")

config_ruim.max_rodadas: 'cinco'
TypeError (longe da causa): can only concatenate str (not "int") to str


Dentro do programa, onde quem cria o objeto é o seu próprio código, isso é aceitável. Mas configuração vem **de fora** (arquivo,
variável de ambiente) — e aí precisa de validação.

**O que falta:** juntar as peças num fluxo único, com configuração vinda do ambiente (validada, com a chave protegida), sem
pagar duas vezes pela mesma pergunta, com logs que não vazam dados, e com testes. ➡️ [10 — pipeline completo](10_pipeline_completo.ipynb)